In [1]:
import sys
import os
from pathlib import Path

# 获取当前 notebook 所在的目录（Jupyter 中）
try:
    # 尝试获取 __file__（在 .py 文件中有效）
    current_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # 在 Jupyter notebook 中，使用当前工作目录
    current_dir = os.getcwd()
    
# 将当前目录添加到系统路径
sys.path.append(current_dir)

from delayed_model import DelayedStateTransitionModel
from utils.helpers import format_state_sequence, save_simulation_result
from base_model import EventType

In [2]:
def demo_delayed_model():
    """演示延迟状态转移模型 - 修正版：部分转移后观测状态为健康"""
    print("=" * 80)
    print("延迟状态转移模型演示")
    print("=" * 80)
    
    # 创建模型实例
    model = DelayedStateTransitionModel(n=2, m=4, k=2)
    
    # 查看模型信息
    info = model.get_model_info()
    print(f"\n📊 模型信息:")
    print(f"   名称: {info['model_name']}")
    print(f"   参数: n={info['model_params']['n']}, m={info['model_params']['m']}, k={info['model_params']['k']}")
    print(f"   初始状态: {info['initial_state']} (不健康)")
    print(f"   可用干预: {info['available_interventions']}")
    
    # 定义干预序列
    interventions = [
        (0, 0, 1),  # step 1: 施加干预
        (0, 0, 1),  # step 2: 继续干预（达到n=2，进入部分转移，观测状态变为健康）
        (0, 0, 0),  # step 3: 停止干预（仍在中间状态持续期内，观测保持健康）
        (0, 0, 0),  # step 4: 仍然没有干预（中间状态持续期结束，观测回到不健康）
        (0, 0, 0),  # step 5: 无干预（保持不健康）
        (0, 0, 1),  # step 6: 重新干预
        (0, 0, 1),  # step 7: 继续干预
        (0, 0, 1),  # step 8: 继续干预
        (0, 0, 1),  # step 9: 继续干预（达到m=4，完全转移到健康）
    ]
    
    print(f"\n📋 干预序列（共 {len(interventions)} 步）:")
    for t, inter in enumerate(interventions, 1):
        print(f"   t={t}: {inter}", end="")
        if inter == (0, 0, 1):
            print(" 🔵 施加干预")
        else:
            print(" ⚪ 无干预")
    
    # 模拟
    result = model.simulate(interventions, return_details=True)
    
    # 格式化输出 - 详细版
    print(f"\n📈 模拟结果（可观测状态序列）:")
    print("-" * 100)
    print(f"{'步数':<6} {'干预向量':<25} {'可观测状态':<25} {'事件类型':<40}")
    print("-" * 100)
    
    for i in range(len(result["states"])):
        if i == 0:
            inter_str = "初始状态"
        else:
            inter_str = str(result["interventions"][i-1])
        
        state = result["states"][i]
        state_str = str(state)
        
        # 添加状态说明
        if state == model.HEALTHY:
            state_desc = "健康"
        else:
            state_desc = "不健康"
        
        display_state = f"{state_str} ({state_desc})"
        
        if i > 0 and "events" in result:
            event = result["events"][i-1]
            # 添加事件的中文说明
            if event == EventType.COMPLETE_TRANSITION:
                event_str = f"{event.value} (完全转移 → 稳定健康)"
            elif event == EventType.PARTIAL_TRANSITION:
                event_str = f"{event.value} (部分转移 → 进入中间状态，观测为健康)"
            elif event == EventType.NO_EFFECTIVE_TRANSITION:
                internal = model.get_internal_state()
                if internal["is_in_intermediate_state"]:
                    event_str = f"{event.value} (中间状态持续期，剩余{internal['partial_transition_counter']}步，观测保持健康)"
                else:
                    event_str = f"{event.value} (无有效干预 → 不健康)"
            else:
                event_str = event.value
        else:
            event_str = "-"
        
        print(f"{i:<6} {inter_str:<25} {display_state:<25} {event_str:<40}")
    
    print("-" * 100)
    
    # 添加状态图例
    print(f"\n📖 可观测状态说明:")
    print(f"   (0, 0, 0) → 健康状态")
    print(f"   (0, 0, 1) → 不健康状态")
    
    # 添加转移规则说明
    print(f"\n📖 状态转移规则:")
    print(f"   1. 连续 {model.model_params['n']} 步干预 → 部分转移")
    print(f"      - 进入不可观测的中间状态，持续 {model.model_params['k']} 步")
    print(f"      - 观测状态变为健康 (0,0,0)")
    print(f"   2. 连续 {model.model_params['m']} 步干预 → 完全转移")
    print(f"      - 直接到达稳定健康状态 (0,0,0)")
    print(f"   3. 无有效干预")
    print(f"      - 如果在中间状态持续期内，观测保持健康")
    print(f"      - 否则，观测状态为不健康 (0,0,1)")
    
    # 保存结果
    save_simulation_result(result, "delayed_model_result.json")
    print(f"\n💾 结果已保存到 delayed_model_result.json")
    
    return model, result

In [3]:
def demo_internal_state_visualization():
    """演示内部状态与可观测状态的对比"""
    print("\n" + "=" * 80)
    print("内部状态 vs 可观测状态对比演示")
    print("=" * 80)
    
    model = DelayedStateTransitionModel(n=2, m=4, k=3)
    
    print(f"\n模型参数: n={model.model_params['n']}, m={model.model_params['m']}, k={model.model_params['k']}")
    print(f"说明: 连续 {model.model_params['n']} 步干预后进入中间状态（内部），持续 {model.model_params['k']} 步")
    print("      在中间状态期间，可观测状态为健康 (0,0,0)")
    
    # 测试序列：部分转移后停止
    interventions = [(0, 0, 1)] * 2 + [(0, 0, 0)] * 5
    
    print(f"\n干预序列: ", end="")
    for i, inter in enumerate(interventions):
        if inter == model.INTERVENTION:
            print(f"{i+1}🔵 ", end="")
        else:
            print(f"{i+1}⚪ ", end="")
    print("\n")
    
    result = model.simulate(interventions, return_details=True)
    
    print(f"{'步数':<6} {'干预':<12} {'可观测状态':<20} {'内部中间状态':<20} {'事件':<25}")
    print("-" * 85)
    
    # 重新模拟以获取内部状态细节
    model.reset_internal_state()
    state = model.get_initial_state()
    
    for i in range(len(interventions) + 1):
        if i == 0:
            inter_str = "初始"
            obs_state = state
            internal_counter = 0
            event_str = "-"
        else:
            history = interventions[:i]
            event = model.check_event(history)
            next_state = model.get_next_state(state, history)
            
            inter_str = "干预" if interventions[i-1] == model.INTERVENTION else "无干预"
            obs_state = next_state
            internal_counter = model._partial_transition_counter
            event_str = event.value
            
            state = next_state
        
        # 格式化输出
        obs_str = "健康" if obs_state == model.HEALTHY else "不健康"
        internal_str = f"剩余{internal_counter}步" if internal_counter > 0 else "无"
        
        print(f"{i:<6} {inter_str:<12} {obs_str:<20} {internal_str:<20} {event_str:<25}")
    
    print("\n💡 关键观察:")
    print("   - 部分转移后（步骤2），可观测状态立即变为健康")
    print("   - 即使停止干预，可观测状态在中间状态持续期内保持健康")
    print("   - 中间状态结束后（步骤5），可观测状态回到不健康")
    print("   - 外部观察者无法直接知道内部是否处于中间状态")

In [4]:
def demo_complete_transition():
    """演示完全转移的稳定性"""
    print("\n" + "=" * 80)
    print("完全转移稳定性演示")
    print("=" * 80)
    
    model = DelayedStateTransitionModel(n=2, m=4, k=2)
    
    # 完全转移后继续观察
    interventions = [(0, 0, 1)] * 4 + [(0, 0, 0)] * 3
    
    print(f"\n干预序列: 4次干预（达到完全转移） + 3次无干预")
    
    result = model.simulate(interventions, return_details=True)
    
    print(f"\n{'步数':<6} {'干预':<15} {'可观测状态':<20} {'事件':<30}")
    print("-" * 75)
    
    for i in range(len(result["states"])):
        if i == 0:
            inter_str = "初始"
            state = result["states"][i]
            state_str = "健康" if state == model.HEALTHY else "不健康"
            event_str = "-"
        else:
            inter = interventions[i-1]
            inter_str = "干预" if inter == model.INTERVENTION else "无干预"
            state = result["states"][i]
            state_str = "健康" if state == model.HEALTHY else "不健康"
            event = result["events"][i-1]
            if event == EventType.COMPLETE_TRANSITION:
                event_str = "完全转移"
            elif event == EventType.PARTIAL_TRANSITION:
                event_str = "部分转移"
            else:
                event_str = "无有效干预"
        
        print(f"{i:<6} {inter_str:<15} {state_str:<20} {event_str:<30}")
    
    print("\n💡 关键观察:")
    print("   - 完全转移后，状态稳定在健康状态")
    print("   - 即使后续停止干预，健康状态也不会回退")
    print("   - 这与部分转移不同（部分转移后停止干预会回退）")

In [5]:
if __name__ == "__main__" or True:
    # 运行主演示
    model, result = demo_delayed_model()
    
    # 运行内部状态对比演示
    demo_internal_state_visualization()
    
    # 运行完全转移演示
    demo_complete_transition()
    
    print("\n" + "=" * 80)
    print("✅ 所有演示完成！")
    print("=" * 80)

延迟状态转移模型演示

📊 模型信息:
   名称: DelayedStateTransitionModel
   参数: n=2, m=4, k=2
   初始状态: (0, 0, 1) (不健康)
   可用干预: [(0, 0, 1), (0, 0, 0)]

📋 干预序列（共 9 步）:
   t=1: (0, 0, 1) 🔵 施加干预
   t=2: (0, 0, 1) 🔵 施加干预
   t=3: (0, 0, 0) ⚪ 无干预
   t=4: (0, 0, 0) ⚪ 无干预
   t=5: (0, 0, 0) ⚪ 无干预
   t=6: (0, 0, 1) 🔵 施加干预
   t=7: (0, 0, 1) 🔵 施加干预
   t=8: (0, 0, 1) 🔵 施加干预
   t=9: (0, 0, 1) 🔵 施加干预

📈 模拟结果（可观测状态序列）:
----------------------------------------------------------------------------------------------------
步数     干预向量                      可观测状态                     事件类型                                    
----------------------------------------------------------------------------------------------------
0      初始状态                      (0, 0, 1) (不健康)           -                                       
1      (0, 0, 1)                 (0, 0, 1) (不健康)           no_effective_transition (无有效干预 → 不健康)   
2      (0, 0, 1)                 (0, 0, 0) (健康)            partial_transition (部分转移 → 进入中间状态，观测为健康)
3      (0